# Stellar density with E(B-V) contours (rubin_sim MAF)

Notebook from https://github.com/knutago/footprint-taskforce/blob/main/footprint_task_force.ipynb

This notebook evaluates `maf.StarDensityMetric` on a full-sky `HealpixSlicer`, overlays E(B-V) contours from `maf.DustMap`, shades the region that passes a joint cut on both, and outlines the Rubin baseline survey footprint and its Wide Fast Deep (WFD) area.

- **Footprint:** `rubin_scheduler.scheduler.utils.get_current_footprint`. This is the scheduler's own region map behind the [baseline WFD](https://survey-strategy.lsst.io/baseline/wfd.html) (about 19.6k deg²). WFD is taken as the same set of regions that MAF's `make_wfd_subset_slicer` uses.

- **Stellar density:** `maf.TrilegalDensityMap` (TRILEGAL, includes the bulge) or `maf.StellarDensityMap` (CatSim/GalFast; disk and halo only, **no bulge**).
- **Dust:** `maf.DustMap` (SFD E(B-V)).

Neither map depends on visits, so no OpSim database is needed. The maps are loaded straight onto the slicer's slice points.

## 1. Environment check

Checks that `rubin_sim` imports and that the map files are where MAF looks for them (`$RUBIN_SIM_DATA_DIR`, or `~/rubin_sim_data` if that isn't set).

If the maps are missing, either point `RUBIN_SIM_DATA_DIR` at an existing copy, or run this in a terminal and then restart the kernel:

```bash
pip install --user rubin-sim
rs_download_data --dirs maps
scheduler_download_data      # footprint inputs: scheduler/dust_maps, Euclid contour
```

In [ ]:
import os
# os.environ["RUBIN_SIM_DATA_DIR"] = "/path/to/rubin_sim_data"   # uncomment to override

import rubin_sim
from rubin_scheduler.data import get_data_dir

print("rubin_sim", rubin_sim.__version__)
data_dir = get_data_dir()
print("data dir:", data_dir)
for sub in ["maps/TriMaps", "maps/StarMaps", "maps/DustMaps", "scheduler/dust_maps"]:
    p = os.path.join(data_dir, sub)
    print(f"  {sub:20s}", "OK" if os.path.isdir(p) else "MISSING")

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import rubin_sim.maf as maf

%matplotlib inline

## 2. Settings

In [ ]:
NSIDE = 64  # native nside of the star-density maps
FILTER = "r"  # u g r i z y
STAR_MODEL = "trilegal"  # "trilegal" (has bulge) or "galfast" (no bulge)
MAG_LIMIT = 17.0  # count stars brighter than this (valid ~15-28)
EBV_LEVELS = [0.12]
COORD = "C"  # plot in equatorial (celestial); use ["C", "G"] for Galactic

# Joint cut: pixels passing BOTH conditions are shaded on the map
STARDENS_CUT = 0.0002  # stars / arcsec^2 (same units as the metric)
STARDENS_SIDE = "below"  # keep pixels "above" or "below" STARDENS_CUT
EBV_CUT = 0.12  # E(B-V) in mag
EBV_SIDE = "below"  # keep pixels "above" or "below" EBV_CUT
CUT_COLOR = "#ff3b8b"

# Rubin baseline footprint (labels from rubin_scheduler's current footprint).
# Default WFD = the regions MAF's make_wfd_subset_slicer uses
WFD_LABELS = ["lowdust", "euclid_overlap", "virgo", "bulgy", "LMC_SMC"]
RESTRICT_CUT_TO_WFD = True  # joint cut counts only pixels inside the WFD
SHOW_FULL_FOOTPRINT = True  # also outline the whole survey footprint (dashed)
WFD_COLOR = "#ff9f1c"
OUTFILE = f"stardens_{STAR_MODEL}_{FILTER}{MAG_LIMIT:g}_ebv.png"

## 3. Load the maps onto the slice points

In [ ]:
slicer = maf.HealpixSlicer(nside=NSIDE, verbose=False)
sp = dict(slicer.slice_points)  # ra, dec (rad), sid, nside, gall, galb

if STAR_MODEL == "trilegal":
    maf.TrilegalDensityMap(filtername=FILTER, nside=NSIDE).run(sp)
else:
    maf.StellarDensityMap(filtername=FILTER).run(sp)
maf.DustMap(nside=NSIDE).run(sp)

print(sorted(k for k in sp if k.startswith(("star", "ebv"))))

## 4. Evaluate `StarDensityMetric` at each pixel

This gives stars per square arcsecond brighter than `MAG_LIMIT`.

In [ ]:
metric = maf.StarDensityMetric(filtername=FILTER, mag_limit=MAG_LIMIT)
lf_key, bin_key = f"starLumFunc_{FILTER}", f"starMapBins_{FILTER}"
stardens = np.full(hp.nside2npix(NSIDE), hp.UNSEEN)
for i in range(stardens.size):
    val = metric.run(None, slice_point={lf_key: sp[lf_key][i], bin_key: sp[bin_key]})
    if val != metric.badval and val > 0:
        stardens[i] = val

ebv = np.asarray(sp["ebv"], dtype=float)
logdens = np.where(stardens > 0, np.log10(stardens), hp.UNSEEN)
print(f"valid pixels: {(stardens > 0).sum()} / {stardens.size}")

## 5. Check coordinate frames and rotate

MAF assumes both maps are stored in equatorial coordinates. This cell checks that from the data: stars and dust both pile up at low |b|, so it tests whether the brightest 5% of pixels sit closer to the Galactic plane when the map is read as equatorial (C) or as Galactic (G). Each map is then rotated into the plotting frame separately.

In [ ]:
def stored_frame(m):
    """Guess whether a full-sky healpix map is stored in equatorial ("C") or
    Galactic ("G") pixel order. Both stars and dust pile up at low |b|, so
    take the brightest 5% of pixels and ask which interpretation puts them
    closer to the Galactic plane."""
    pix = np.arange(m.size)
    good = np.isfinite(m) & (m != hp.UNSEEN)
    top = good & (m >= np.percentile(m[good], 95))
    theta, phi = hp.pix2ang(NSIDE, pix[top])
    b_if_G = 90.0 - np.degrees(theta)  # pixel lat is b
    b_if_C = 90.0 - np.degrees(hp.Rotator(coord=["C", "G"])(theta, phi)[0])
    return "C" if np.median(np.abs(b_if_C)) < np.median(np.abs(b_if_G)) else "G"


def to_frame(m, src, dst):
    """Rotate a healpix map from frame src to dst, preserving UNSEEN pixels."""
    if src == dst:
        return m
    rot = hp.Rotator(coord=[src, dst])
    bad = rot.rotate_map_pixel((m == hp.UNSEEN).astype(float)) > 0.5
    out = rot.rotate_map_pixel(np.where(m == hp.UNSEEN, 0.0, m))
    out[bad] = hp.UNSEEN
    return out


target = COORD[-1]
star_frame, dust_frame = stored_frame(logdens), stored_frame(ebv)
print(f"stellar density stored in {star_frame}, E(B-V) stored in {dust_frame}")
logdens_p = to_frame(logdens, star_frame, target)
ebv_p = to_frame(ebv, dust_frame, target)
frame = {"C": "Equatorial", "G": "Galactic"}[target]

## 6. Rubin survey footprint and WFD

Loads the scheduler's current footprint. Every pixel carries a region label (`lowdust`, `euclid_overlap`, `virgo`, `bulgy`, `LMC_SMC`, `nes`, `dusty_plane`, `scp`, or empty if outside the survey). The WFD is the union of `WFD_LABELS`.

For reference, the low-dust WFD in that code is roughly: smoothed SFD E(B-V) < 0.199, −70° < Dec < +15° (upper limit lowered by 12° where b < 0), plus the Euclid-overlap extension. The WFD-level "bulgy" Galactic plane/bulge is a set of circles, the largest centred on (RA, Dec) = (266.3°, −29°) with radius 17°. Check the code for the exact current values; the footprint map is in equatorial coordinates.

In [ ]:
from rubin_scheduler.scheduler.utils import get_current_footprint

_, fp_labels = get_current_footprint(nside=NSIDE)
pix_area = hp.nside2pixarea(NSIDE, degrees=True)

labels, counts = np.unique(fp_labels[fp_labels != ""], return_counts=True)
for lab, n in zip(labels, counts):
    print(f"  {lab:15s} {n * pix_area:8,.0f} deg^2" + ("  (WFD)" if lab in WFD_LABELS else ""))

wfd = np.isin(fp_labels, WFD_LABELS)
survey = fp_labels != ""
print(f"WFD total: {wfd.sum() * pix_area:,.0f} deg^2;  full footprint: {survey.sum() * pix_area:,.0f} deg^2")

# Footprint is equatorial; rotate into the plotting frame like the other maps
wfd_p = to_frame(wfd.astype(float), "C", target) > 0.5
survey_p = to_frame(survey.astype(float), "C", target) > 0.5

## 7. Joint cut on stellar density and E(B-V)

Selects the pixels that pass both the stellar-density cut and the E(B-V) cut set in **Settings**, and reports their sky area. With `RESTRICT_CUT_TO_WFD = True`, only pixels inside the WFD are kept. The selection is made on the maps after they've been rotated into the plotting frame, so it lines up with the plot.

In [ ]:
def side_mask(m, cut, side):
    good = np.isfinite(m) & (m != hp.UNSEEN)
    if side == "above":
        return good & (m >= cut)
    if side == "below":
        return good & (m <= cut)
    raise ValueError(f"side must be 'above' or 'below', got {side!r}")


# logdens_p is log10(stars/arcsec^2), so compare against log10 of the cut
sel_all = side_mask(logdens_p, np.log10(STARDENS_CUT), STARDENS_SIDE) & side_mask(ebv_p, EBV_CUT, EBV_SIDE)
sel = sel_all & wfd_p if RESTRICT_CUT_TO_WFD else sel_all

sym = {"above": ">=", "below": "<="}
cut_label = (
    f"n* {sym[STARDENS_SIDE]} {STARDENS_CUT:g} arcsec$^{{-2}}$ & " f"E(B-V) {sym[EBV_SIDE]} {EBV_CUT:g}"
)
if RESTRICT_CUT_TO_WFD:
    cut_label += " (in WFD)"
print(f"all sky:  {sel_all.sum() * pix_area:,.0f} deg^2 pass the cut")
print(
    f"in WFD:   {(sel_all & wfd_p).sum() * pix_area:,.0f} deg^2 "
    f"({100 * (sel_all & wfd_p).sum() / max(wfd_p.sum(), 1):.1f}% of WFD)"
)
print(
    f"{cut_label.replace('$^{-2}$', '^-2')}: {sel.sum()} pixels = {sel.sum() * pix_area:,.0f} deg^2 "
    f"({100 * sel.mean():.1f}% of sky)"
)

## 8. Plot stellar density, E(B-V) contours, joint cut and footprint

In [ ]:
fig = plt.figure(figsize=(11, 6.5))
hp.mollview(
    logdens_p,
    fig=fig.number,
    cmap="viridis",
    title=f"Stellar density ({FILTER} < {MAG_LIMIT:g}, {STAR_MODEL}) with E(B-V) contours [{frame}]",
    unit=r"log$_{10}$ stars arcsec$^{-2}$",
)
hp.graticule(dpar=30, dmer=60, alpha=0.4)

# Project E(B-V) onto the same Mollweide grid and contour it on the same axes
proj = hp.projector.MollweideProj(xsize=1600)
img = proj.projmap(ebv_p, lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))
img = np.ma.masked_where(~np.isfinite(img) | (img == hp.UNSEEN), img)
x0, x1, y0, y1 = proj.get_extent()
X, Y = np.meshgrid(np.linspace(x0, x1, img.shape[1]), np.linspace(y0, y1, img.shape[0]))

ax = plt.gca()
cs = ax.contour(X, Y, img, levels=EBV_LEVELS, colors="white", linewidths=0.8)
ax.clabel(cs, fmt="%.2f", fontsize=7)

# Joint-cut layer: translucent fill plus a solid outline
selimg = proj.projmap(sel.astype(float), lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))
selimg = np.ma.masked_where(~np.isfinite(selimg) | (selimg == hp.UNSEEN), selimg)
ax.contourf(X, Y, selimg, levels=[0.5, 1.5], colors=[CUT_COLOR], alpha=0.35)
ax.contour(X, Y, selimg, levels=[0.5], colors=[CUT_COLOR], linewidths=1.2)


# Footprint outlines: WFD solid, full survey footprint dashed
def outline(mask, **kw):
    im = proj.projmap(mask.astype(float), lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))
    im = np.ma.masked_where(~np.isfinite(im) | (im == hp.UNSEEN), im)
    plt.gca().contour(X, Y, im, levels=[0.5], **kw)


outline(wfd_p, colors=[WFD_COLOR], linewidths=1.8)
if SHOW_FULL_FOOTPRINT:
    outline(survey_p, colors=["0.85"], linewidths=1.0, linestyles="--")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

handles = [
    Patch(facecolor=CUT_COLOR, edgecolor=CUT_COLOR, alpha=0.5, label=cut_label),
    Line2D([], [], color="white", lw=0.8, label="E(B-V) contours"),
    Line2D([], [], color=WFD_COLOR, lw=1.8, label="WFD footprint"),
]
if SHOW_FULL_FOOTPRINT:
    handles.append(Line2D([], [], color="0.85", lw=1.0, ls="--", label="full survey footprint"))
ax.legend(
    handles=handles,
    loc="lower left",
    bbox_to_anchor=(0.0, -0.02),
    fontsize=8,
    framealpha=0.8,
    facecolor="0.3",
    labelcolor="white",
)

fig.savefig(OUTFILE, dpi=150, bbox_inches="tight")
print("wrote", OUTFILE)
plt.show()

## 9. Compare with an external HEALPix map

Reads an external HEALPix map, by default `rubin_wfd_stellar_masked_nside64.fits` (column `wfd_usable_fraction`, the usable fraction of each WFD pixel). It is compared with the maps above in four ways:

1. **Footprint:** area covered by the external map vs our WFD.
2. **Drivers:** the external value against stellar density and E(B-V), with Spearman rank correlations.
3. **Cut agreement:** our joint cut (section 7) against the external map thresholded at `EXT_MIN`, as a confusion table and an agreement map.
4. **Effective area:** our cut area weighted by the external fraction.

All statistics are computed in equatorial pixels (the native frame of the footprint and the external map). The maps are rotated only for display. The file's `COORDSYS` and `NSIDE` header keywords are read and honoured.

In [ ]:
from astropy.io import fits
from scipy.stats import spearmanr

EXT_MAP_FILE = "rubin_wfd_stellar_masked_nside64.fits"
EXT_FIELD = 0  # column index (or name) to read
EXT_LABEL = "WFD usable fraction"
EXT_MIN = 0.5  # threshold that counts a pixel as "good" in the external map

hdr = fits.getheader(EXT_MAP_FILE, 1)
ext_coord = hdr.get("COORDSYS", "C").strip()[0].upper()
ext_coord = {"E": "E", "G": "G"}.get(ext_coord, "C")  # 'C'/'Q' -> equatorial
ext = hp.read_map(EXT_MAP_FILE, field=EXT_FIELD)  # always returned in RING order
if hp.get_nside(ext) != NSIDE:
    ext = hp.ud_grade(ext, NSIDE)
print(f"{EXT_MAP_FILE}: nside={hdr.get('NSIDE')}, ordering={hdr.get('ORDERING')}, coordsys={ext_coord}")


def to_frame_nearest(m, src, dst):
    """Rotate by nearest-pixel lookup (no interpolation): good for masks and labels."""
    if src == dst:
        return m
    th, ph = hp.pix2ang(NSIDE, np.arange(m.size))
    th2, ph2 = hp.Rotator(coord=[dst, src])(th, ph)
    return m[hp.ang2pix(NSIDE, th2, ph2)]


# Everything in equatorial pixels for the statistics
ext_c = to_frame_nearest(ext, ext_coord, "C")
ext_ok = np.isfinite(ext_c) & (ext_c != hp.UNSEEN)
logdens_c = to_frame_nearest(logdens, star_frame, "C")
ebv_c = to_frame_nearest(ebv, dust_frame, "C")
sel_all_c = side_mask(logdens_c, np.log10(STARDENS_CUT), STARDENS_SIDE) & side_mask(ebv_c, EBV_CUT, EBV_SIDE)
sel_c = sel_all_c & wfd if RESTRICT_CUT_TO_WFD else sel_all_c

print(
    f"external map covers {ext_ok.sum() * pix_area:,.0f} deg^2; "
    f"values {ext_c[ext_ok].min():.3g} to {ext_c[ext_ok].max():.3g}; "
    f"weighted area {ext_c[ext_ok].sum() * pix_area:,.0f} deg^2"
)

### 9a. Footprint: external coverage vs our WFD

In [ ]:
both = ext_ok & wfd
print(f"in both:            {both.sum() * pix_area:8,.0f} deg^2")
print(f"external only:      {(ext_ok & ~wfd).sum() * pix_area:8,.0f} deg^2")
print(f"our WFD only:       {(~ext_ok & wfd).sum() * pix_area:8,.0f} deg^2")
print(f"Jaccard overlap:    {both.sum() / max((ext_ok | wfd).sum(), 1):.3f}")

# External-only pixels by footprint label: shows which regions differ
lab, cnt = np.unique(fp_labels[ext_ok & ~wfd], return_counts=True)
for l, n in zip(lab, cnt):
    print(f"  external-only in {l or '(outside footprint)':22s} {n * pix_area:7,.0f} deg^2")

### 9b. External map vs stellar density and E(B-V)

In [ ]:
good = ext_ok & (logdens_c != hp.UNSEEN) & np.isfinite(ebv_c) & (ebv_c > 0)
x1, x2, y = logdens_c[good], np.log10(ebv_c[good]), ext_c[good]
r1, r2 = spearmanr(x1, y).statistic, spearmanr(x2, y).statistic

fig2, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for a, x, xl, r, cut, side in [
    (axs[0], x1, r"log$_{10}$ stars arcsec$^{-2}$", r1, np.log10(STARDENS_CUT), STARDENS_SIDE),
    (axs[1], x2, r"log$_{10}$ E(B-V)", r2, np.log10(EBV_CUT), EBV_SIDE),
]:
    hb = a.hexbin(x, y, gridsize=60, mincnt=1, bins="log", cmap="viridis")
    a.axvline(cut, color=CUT_COLOR, lw=1.5, label=f"cut ({side})")
    a.axhline(EXT_MIN, color="0.4", lw=1, ls="--", label=f"EXT_MIN = {EXT_MIN:g}")
    a.set_xlabel(xl)
    a.set_title(f"Spearman $\\rho$ = {r:+.2f}", fontsize=10)
    a.legend(fontsize=8, loc="lower left")
axs[0].set_ylabel(EXT_LABEL)
fig2.colorbar(hb, ax=axs, label="pixels (log)", shrink=0.85)
plt.show()

### 9c. Our joint cut vs the external map above `EXT_MIN`

In [ ]:
ext_good = ext_ok & (ext_c >= EXT_MIN)
a_both, a_cut, a_ext = (sel_c & ext_good), (sel_c & ~ext_good), (~sel_c & ext_good)
print(f"external >= {EXT_MIN:g}:          {ext_good.sum() * pix_area:8,.0f} deg^2")
print(f"our cut:                  {sel_c.sum() * pix_area:8,.0f} deg^2")
print(f"  both:                   {a_both.sum() * pix_area:8,.0f} deg^2")
print(f"  our cut only:           {a_cut.sum() * pix_area:8,.0f} deg^2")
print(f"  external only:          {a_ext.sum() * pix_area:8,.0f} deg^2")
print(f"Jaccard overlap:          {a_both.sum() / max((sel_c | ext_good).sum(), 1):.3f}")
print(
    f"mean external value inside our cut: {ext_c[sel_c & ext_ok].mean():.3f}; "
    f"outside it (in external map): {ext_c[~sel_c & ext_ok].mean():.3f}"
)
print(
    f"effective area of our cut weighted by external map: "
    f"{ext_c[sel_c & ext_ok].sum() * pix_area:,.0f} deg^2"
)

# Agreement map: 1 = both, 2 = our cut only, 3 = external only
agree = np.full(ext_c.size, hp.UNSEEN)
agree[a_both], agree[a_cut], agree[a_ext] = 1, 2, 3
agree_p = to_frame_nearest(agree, "C", target)
ext_p = to_frame_nearest(np.where(ext_ok, ext_c, hp.UNSEEN), "C", target)

### 9d. Maps: external map, and where it agrees with our cut

In [ ]:
from matplotlib.colors import ListedColormap

AGREE_COLORS = ["#2a9d8f", CUT_COLOR, "#e9c46a"]  # both, our cut only, external only

fig3 = plt.figure(figsize=(16, 5))
hp.mollview(
    ext_p, fig=fig3.number, sub=(1, 2, 1), cmap="viridis", title=f"{EXT_LABEL} [{frame}]", unit=EXT_LABEL
)
hp.graticule(dpar=30, dmer=60, alpha=0.3)
outline(wfd_p, colors=[WFD_COLOR], linewidths=1.5)
outline(sel, colors=[CUT_COLOR], linewidths=1.0)
plt.gca().legend(
    handles=[
        Line2D([], [], color=WFD_COLOR, lw=1.5, label="our WFD"),
        Line2D([], [], color=CUT_COLOR, lw=1.0, label="our joint cut"),
    ],
    loc="lower left",
    fontsize=8,
)

hp.mollview(
    agree_p,
    fig=fig3.number,
    sub=(1, 2, 2),
    cmap=ListedColormap(AGREE_COLORS),
    min=0.5,
    max=3.5,
    cbar=False,
    title=f"Our cut vs {EXT_LABEL} >= {EXT_MIN:g} [{frame}]",
)
hp.graticule(dpar=30, dmer=60, alpha=0.3)
plt.gca().legend(
    handles=[
        Patch(color=c, label=l)
        for c, l in zip(AGREE_COLORS, ["both", "our cut only", f"external >= {EXT_MIN:g} only"])
    ],
    loc="lower left",
    fontsize=8,
)
fig3.savefig(OUTFILE.replace(".png", "_vs_external.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10. Modified footprint

Builds a new footprint from the scheduler's current one, in five steps. All of them work in equatorial pixels, the native frame of the footprint.

1. **Cut:** `lowdust` pixels that pass the joint cut stay `lowdust` (the cut is applied without the WFD restriction). Labels in `CLAIM_FROM_LABELS` that pass the cut are also turned into `lowdust`.
2. **Smooth:** the `lowdust` edge is smoothed with a Gaussian of FWHM `LOWDUST_SMOOTH_FWHM` (degrees; 0 turns it off). This uses normalised convolution over `lowdust` plus the regions it may trade with (`HOLE_FILL_LABELS`), so edges against empty sky or other WFD regions are left alone. `lowdust` can only grow into `HOLE_FILL_LABELS` pixels.
3. **Fill holes:** `lowdust` pixels lost to the cut or the smoothing are grouped into connected holes. Each hole goes to the nearest region in `HOLE_FILL_LABELS`: the one it shares the most pixel edges with, or, if it is enclosed, the one whose nearest pixel is closest. `HOLE_MODE = "pixel"` assigns each pixel separately instead.
4. **Squash bulgy islands:** connected pieces of `bulgy` smaller than `BULGY_ISLAND_MAX_AREA` deg² are given to the nearest region the same way. In the current footprint these are the two 3°-radius circles toward the anticentre, at RA, Dec ≈ (100.9°, +9.6°) and (84.9°, −5.7°).
   **Trim bulgy:** `BULGY_TRIM_FRAC` (default 10%) of the remaining `bulgy` area is then peeled off its outer edge: the pixels closest to a `HOLE_FILL_LABELS` region go first, and each goes to the nearest such region. Edges shared with `lowdust` or other WFD regions are not trimmed. This runs before the bridge, so the bridge still reaches the new edge.
5. **Bridge:** a Galactic-coordinate box on the southern (b < 0) side of the bulge, which is the left side in the equatorial plot, is relabelled `BRIDGE_LABEL`. Only pixels whose labels are in `BRIDGE_TAKE_FROM` are changed, so the box can be drawn generously: it fills only the gap between `bulgy` and `lowdust`. The cell checks that `bulgy` and `lowdust` end up connected.

The table then reports area and pointings per component, at `POINTINGS_PER_DEG2` = 1.3 pointings per 9.6 deg².

In [ ]:
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

POINTINGS_PER_DEG2 = 1.3 / 9.6
NEW_FP_FILE = f"footprint_labels_cut_{STAR_MODEL}_{FILTER}{MAG_LIMIT:g}.fits"

# 1. cut
CLAIM_FROM_LABELS = []  # other labels the cut may convert to lowdust
# 2. smoothing
LOWDUST_SMOOTH_FWHM = 5.0  # deg; 0 = no smoothing
# 3. hole filling
HOLE_FILL_LABELS = ["dusty_plane", "nes", "scp"]  # regions that can absorb lowdust holes
HOLE_MODE = "component"  # "component": whole hole -> one region; "pixel"
# 4. bulgy islands
BULGY_ISLAND_MAX_AREA = 60.0  # deg^2; bulgy pieces smaller than this are removed
BULGY_TRIM_FRAC = 0.10  # then trim this fraction of bulgy area off its edge (0 = off)
# 5. bridge (Galactic box, degrees; l may wrap through 0)
BRIDGE_L = (-6.0, 26.0)  # l range, as (min, max) in [-180, 180]
BRIDGE_B = (-45.0, -5.0)  # b range
BRIDGE_TAKE_FROM = ["dusty_plane"]  # only these labels become bridge
BRIDGE_LABEL = "bridge"
BRIDGE_IN_WFD = True  # count the bridge as WFD in the totals

npix = hp.nside2npix(NSIDE)
NBRS = hp.get_all_neighbours(NSIDE, np.arange(npix))  # (8, npix), -1 = none


def to_frame_nearest(m, src, dst):
    """Rotate by nearest-pixel lookup (no interpolation): good for masks and labels."""
    if src == dst:
        return m
    th, ph = hp.pix2ang(NSIDE, np.arange(m.size))
    th2, ph2 = hp.Rotator(coord=[dst, src])(th, ph)
    return m[hp.ang2pix(NSIDE, th2, ph2)]


def components(mask):
    """Connected groups (8-neighbour) of True pixels: returns (pixels, group id per pixel)."""
    pix = np.where(mask)[0]
    idx = -np.ones(npix, dtype=int)
    idx[pix] = np.arange(pix.size)
    nb = NBRS[:, pix]
    ok = (nb >= 0) & mask[np.clip(nb, 0, None)]
    r = np.broadcast_to(np.arange(pix.size), nb.shape)[ok]
    adj = coo_matrix((np.ones(r.size), (r, idx[nb[ok]])), shape=(pix.size, pix.size))
    return pix, connected_components(adj, directed=False)[1]


def fill_nearest(labels, mask, cand_labels, mode="component"):
    """Relabel the pixels in mask with the nearest region in cand_labels.
    Returns (pixels, assigned labels, group ids)."""
    cand = np.isin(labels, cand_labels) & ~mask
    if not mask.any():
        return np.array([], int), np.array([], "U20"), np.array([], int)
    if not cand.any():
        raise ValueError(f"no pixels carry any of {cand_labels}")
    cpix = np.where(cand)[0]
    tree = cKDTree(np.array(hp.pix2vec(NSIDE, cpix)).T)
    if mode == "pixel":
        pix = np.where(mask)[0]
        _, j = tree.query(np.array(hp.pix2vec(NSIDE, pix)).T)
        out, gid = labels[cpix[j]], np.arange(pix.size)
    else:
        pix, gid = components(mask)
        out = np.empty(pix.size, dtype="U20")
        for g in np.unique(gid):
            members = pix[gid == g]
            touch = NBRS[:, members].ravel()
            touch = touch[(touch >= 0)]
            touch = touch[cand[touch]]
            if touch.size:  # touches candidate regions: most shared edges
                labs, cnt = np.unique(labels[touch], return_counts=True)
                out[gid == g] = labs[np.argmax(cnt)]
            else:  # enclosed: closest candidate pixel
                d, j = tree.query(np.array(hp.pix2vec(NSIDE, members)).T)
                out[gid == g] = labels[cpix[j[np.argmin(d)]]]
    labels[pix] = out
    return pix, out, gid


def report(title, out, gid, cand_labels):
    for l in cand_labels:
        sel_ = out == l
        if sel_.any():
            print(
                f"    {title} -> {l:14s} {sel_.sum() * pix_area:8,.0f} deg^2"
                f"  ({np.unique(gid[sel_]).size} pieces)"
            )


old_labels = np.asarray(fp_labels).astype("U20")
new_labels = old_labels.copy()

# ---- 1. cut ----
cut_c = side_mask(
    to_frame_nearest(logdens, star_frame, "C"), np.log10(STARDENS_CUT), STARDENS_SIDE
) & side_mask(to_frame_nearest(ebv, dust_frame, "C"), EBV_CUT, EBV_SIDE)
is_low = old_labels == "lowdust"
claim = np.isin(old_labels, CLAIM_FROM_LABELS) & cut_c
low = (is_low & cut_c) | claim
print(
    f"1. cut:     lowdust kept {(is_low & cut_c).sum() * pix_area:,.0f} deg^2, "
    f"failed {(is_low & ~cut_c).sum() * pix_area:,.0f} deg^2, "
    f"claimed {claim.sum() * pix_area:,.0f} deg^2 (from {CLAIM_FROM_LABELS or 'none'})"
)

# ---- 2. smooth the lowdust edge (normalised convolution) ----
tradeable = np.isin(old_labels, HOLE_FILL_LABELS) | is_low | claim
if LOWDUST_SMOOTH_FWHM > 0:
    import warnings

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        num = hp.smoothing(low.astype(float), fwhm=np.radians(LOWDUST_SMOOTH_FWHM))
        den = hp.smoothing(tradeable.astype(float), fwhm=np.radians(LOWDUST_SMOOTH_FWHM))
    frac = np.where(den > 0.05, num / np.maximum(den, 1e-6), 0.0)
    smooth_low = tradeable & (frac >= 0.5)
    gained = smooth_low & ~low & np.isin(old_labels, HOLE_FILL_LABELS + ["lowdust"])
    lost = low & ~smooth_low
    low = (low & ~lost) | gained
    print(
        f"2. smooth:  FWHM {LOWDUST_SMOOTH_FWHM:g} deg, lowdust gained {gained.sum() * pix_area:,.0f} deg^2, "
        f"lost {lost.sum() * pix_area:,.0f} deg^2"
    )
else:
    print("2. smooth:  off")

# ---- 3. fill holes left in the old lowdust area ----
new_labels[low] = "lowdust"
holes = is_low & ~low
new_labels[holes] = ""  # temporarily unassigned
pix, out, gid = fill_nearest(new_labels, holes, HOLE_FILL_LABELS, HOLE_MODE)
print(f"3. holes:   {holes.sum() * pix_area:,.0f} deg^2 in {np.unique(gid).size} pieces")
report("hole", out, gid, HOLE_FILL_LABELS)

# ---- 4. squash small bulgy islands ----
bpix, bgid = components(new_labels == "bulgy")
sizes = np.bincount(bgid) * pix_area if bpix.size else np.array([])
small = np.isin(bgid, np.where(sizes < BULGY_ISLAND_MAX_AREA)[0])
islands = np.zeros(npix, bool)
islands[bpix[small]] = True
print(
    f"4. bulgy:   {sizes.size} pieces, sizes (deg^2): "
    + ", ".join(f"{a:,.0f}" for a in sorted(sizes, reverse=True))
)
if islands.any():
    new_labels[islands] = ""
    _, out, gid = fill_nearest(new_labels, islands, HOLE_FILL_LABELS, "component")
    print(
        f"           removed {islands.sum() * pix_area:,.0f} deg^2 in {np.unique(gid).size} islands "
        f"< {BULGY_ISLAND_MAX_AREA:g} deg^2"
    )
    report("island", out, gid, HOLE_FILL_LABELS)

# ---- 4b. trim the bulgy edge by BULGY_TRIM_FRAC of its area ----
if BULGY_TRIM_FRAC > 0:
    bul = np.where(new_labels == "bulgy")[0]
    toward = np.where(np.isin(new_labels, HOLE_FILL_LABELS))[0]
    n_trim = int(round(BULGY_TRIM_FRAC * bul.size))
    if n_trim and toward.size:
        # distance from each bulgy pixel to the nearest pixel it can be trimmed into;
        # edges shared with lowdust / other WFD regions are not trimmed
        d, _ = cKDTree(np.array(hp.pix2vec(NSIDE, toward)).T).query(np.array(hp.pix2vec(NSIDE, bul)).T)
        trim = np.zeros(npix, bool)
        trim[bul[np.argsort(d, kind="stable")[:n_trim]]] = True
        new_labels[trim] = ""
        _, out, gid = fill_nearest(new_labels, trim, HOLE_FILL_LABELS, "pixel")
        a0 = (old_labels == "bulgy").sum() * pix_area
        a1 = (new_labels == "bulgy").sum() * pix_area
        print(
            f"4b. trim:   removed {trim.sum() * pix_area:,.0f} deg^2 from the bulgy edge "
            f"({100 * BULGY_TRIM_FRAC:g}% of {bul.size * pix_area:,.0f}); bulgy now {a1:,.0f} deg^2 "
            f"({100 * (a1 / a0 - 1):+.1f}% vs current footprint)"
        )
        for l in HOLE_FILL_LABELS:
            if (out == l).any():
                print(f"    trim -> {l:14s} {(out == l).sum() * pix_area:8,.0f} deg^2")
        n_pieces = np.unique(components(new_labels == "bulgy")[1]).size
        print(f"           bulgy now in {n_pieces} pieces")

# ---- 5. bridge between bulgy and lowdust, south of the bulge ----
th, ph = hp.pix2ang(NSIDE, np.arange(npix))
gth, gph = hp.Rotator(coord=["C", "G"])(th, ph)
gl = (np.degrees(gph) + 180.0) % 360.0 - 180.0
gb = 90.0 - np.degrees(gth)
in_box = (gl >= BRIDGE_L[0]) & (gl <= BRIDGE_L[1]) & (gb >= BRIDGE_B[0]) & (gb <= BRIDGE_B[1])
bridge = in_box & np.isin(new_labels, BRIDGE_TAKE_FROM)
new_labels[bridge] = BRIDGE_LABEL
print(
    f"5. bridge:  {bridge.sum() * pix_area:,.0f} deg^2 relabelled {BRIDGE_LABEL!r} "
    f"(box l={BRIDGE_L}, b={BRIDGE_B})"
)

# connectivity check: is the main bulgy piece now joined to lowdust?
j_pix, j_gid = components(np.isin(new_labels, ["bulgy", "lowdust", BRIDGE_LABEL]))
gid_of = np.full(npix, -1)
gid_of[j_pix] = j_gid
bul = np.where(new_labels == "bulgy")[0]
main_bulge = np.bincount(gid_of[bul]).argmax() if bul.size else -1
connected = (gid_of[new_labels == "lowdust"] == main_bulge).any()
print(f"           bulgy connected to lowdust: {'YES' if connected else 'NO - widen BRIDGE_B / BRIDGE_L'}")

### 10b. Area and pointings per component

In [ ]:
# Area and pointings per component
comps = [
    l
    for l in [
        "lowdust",
        "euclid_overlap",
        "virgo",
        "LMC_SMC",
        "bulgy",
        BRIDGE_LABEL,
        "dusty_plane",
        "nes",
        "scp",
    ]
    if l in set(old_labels) | set(new_labels)
]
comps += sorted((set(old_labels) | set(new_labels)) - set(comps) - {""})
rows = []
for c in comps:
    a_old = (old_labels == c).sum() * pix_area
    a_new = (new_labels == c).sum() * pix_area
    rows.append(
        {
            "component": c,
            "WFD": c in WFD_LABELS or (c == BRIDGE_LABEL and BRIDGE_IN_WFD),
            "area_current_deg2": a_old,
            "area_new_deg2": a_new,
            "change_deg2": a_new - a_old,
            "pointings_current": a_old * POINTINGS_PER_DEG2,
            "pointings_new": a_new * POINTINGS_PER_DEG2,
        }
    )
base = pd.DataFrame(rows)
num = base.drop(columns=["component", "WFD"])
totals = pd.DataFrame(
    [
        {"component": "WFD total", "WFD": True, **num[base.WFD].sum().to_dict()},
        {"component": "non-WFD total", "WFD": False, **num[~base.WFD].sum().to_dict()},
        {"component": "footprint total", "WFD": None, **num.sum().to_dict()},
    ]
)
tab = pd.concat([base, totals], ignore_index=True)

tab.to_csv(NEW_FP_FILE.replace(".fits", "_areas.csv"), index=False)
display(
    tab.style.format({c: "{:,.0f}" for c in tab.columns if c not in ("component", "WFD")}).hide(axis="index")
)

### 10c. Visit budget

Total visits per component = pointings × median visits per pointing, for both the current and the modified footprint. `bridge` gets the WFD value unless set separately. For scale, `rubin_scheduler` budgets about 2.2 million visits for a 10-year survey (`Footprint.estimate_counts`).

In [ ]:
WFD_VISITS = 750
VISITS = {
    "lowdust": WFD_VISITS,
    "bulgy": WFD_VISITS,
    "LMC_SMC": WFD_VISITS,
    "euclid_overlap": WFD_VISITS,
    "virgo": WFD_VISITS,
    BRIDGE_LABEL: WFD_VISITS,
    "dusty_plane": 170,
    "nes": 150,
    "scp": 100,
}

vt = base[["component", "WFD", "pointings_current", "pointings_new"]].copy()
vt["median_visits"] = vt.component.map(VISITS)
missing = vt.loc[vt.median_visits.isna(), "component"].tolist()
if missing:
    print("no visit count set for:", missing, "(counted as 0)")
vt["median_visits"] = vt.median_visits.fillna(0)
vt["visits_current"] = vt.pointings_current * vt.median_visits
vt["visits_new"] = vt.pointings_new * vt.median_visits
vt["change"] = vt.visits_new - vt.visits_current
cols = ["pointings_current", "pointings_new", "visits_current", "visits_new", "change"]
vt = pd.concat(
    [
        vt,
        pd.DataFrame(
            [
                {"component": "WFD total", "WFD": True, **vt.loc[vt.WFD, cols].sum().to_dict()},
                {"component": "non-WFD total", "WFD": False, **vt.loc[~vt.WFD, cols].sum().to_dict()},
                {"component": "TOTAL", "WFD": None, **vt[cols].sum().to_dict()},
            ]
        ),
    ],
    ignore_index=True,
)

vt.to_csv(NEW_FP_FILE.replace(".fits", "_visits.csv"), index=False)
display(vt.style.format({c: "{:,.0f}" for c in cols + ["median_visits"]}, na_rep="").hide(axis="index"))
tot = vt.iloc[-1]
print(
    f"total visits: current footprint {tot.visits_current:,.0f}, modified footprint {tot.visits_new:,.0f} "
    f"({tot.change:+,.0f})"
)

In [ ]:
# Map of the modified footprint, plus a FITS file of the new labels
order = [c for c in comps if (new_labels == c).any()]
code = np.full(new_labels.size, hp.UNSEEN)
for i, c in enumerate(order):
    code[new_labels == c] = i
code_p = to_frame_nearest(code, "C", target)

palette = [
    "#2a9d8f",
    "#e9c46a",
    "#f4a261",
    "#8ab17d",
    "#e76f51",
    "#6d597a",
    "#457b9d",
    "#a8dadc",
    "#b5838d",
    "#ffb4a2",
]
colors = (palette * 2)[: len(order)]

fig4 = plt.figure(figsize=(11, 6.5))
hp.mollview(
    code_p,
    fig=fig4.number,
    cmap=ListedColormap(colors),
    min=-0.5,
    max=len(order) - 0.5,
    cbar=False,
    title=f"Modified footprint: joint cut replaces lowdust [{frame}]",
)
hp.graticule(dpar=30, dmer=60, alpha=0.3)
outline(to_frame_nearest(old_labels == "lowdust", "C", target), colors=["k"], linewidths=0.8, linestyles=":")
handles = [
    Patch(color=col, label=f"{c}: {(new_labels == c).sum() * pix_area:,.0f} deg$^2$")
    for c, col in zip(order, colors)
]
handles.append(Line2D([], [], color="k", ls=":", lw=0.8, label="current lowdust"))

# Ecliptic (lat 0, solid) plus optional parallel lines of ecliptic latitude (dashed)
ECLIPTIC_LATS = [0.0]  # e.g. [-10.0, 0.0, 10.0] to show the NES +/-10 deg band
ECLIPTIC_COLOR = "#ffd23f"
elon = np.linspace(0.0, 360.0, 721)
for elat in ECLIPTIC_LATS:
    eth, eph = np.radians(90.0 - elat) * np.ones_like(elon), np.radians(elon)
    if target != "E":
        eth, eph = hp.Rotator(coord=["E", target])(eth, eph)
    hp.projplot(eth, eph, color=ECLIPTIC_COLOR, lw=1.6 if elat == 0 else 1.0, ls="-" if elat == 0 else "--")
handles.append(Line2D([], [], color=ECLIPTIC_COLOR, lw=1.6, label="ecliptic"))
if any(l != 0 for l in ECLIPTIC_LATS):
    handles.append(
        Line2D(
            [],
            [],
            color=ECLIPTIC_COLOR,
            lw=1.0,
            ls="--",
            label="ecliptic lat " + ", ".join(f"{l:+g}" for l in ECLIPTIC_LATS if l != 0) + " deg",
        )
    )
plt.gca().legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8, framealpha=0.9)
fig4.savefig(NEW_FP_FILE.replace(".fits", ".png"), dpi=150, bbox_inches="tight")
plt.show()

# Integer label map (equatorial, RING) with the code -> label key in the header
lab_int = np.full(new_labels.size, -1, dtype=np.int32)
for i, c in enumerate(order):
    lab_int[new_labels == c] = i
hp.write_map(
    NEW_FP_FILE,
    lab_int,
    coord="C",
    column_names=["label"],
    dtype=np.int32,
    extra_header=[(f"LABEL{i}", c) for i, c in enumerate(order)],
    overwrite=True,
)
print("wrote", NEW_FP_FILE, "and", NEW_FP_FILE.replace(".fits", "_areas.csv"))